#IMPORTING DATA

In [ ]:
import kagglehub
# Download latest version
path = kagglehub.dataset_download("kazanova/sentiment140")
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'sentiment140' dataset.
Path to dataset files: /kaggle/input/sentiment140


In [ ]:
import pandas as pd
import os

# Construct path to the downloaded dataset
csv_path = os.path.join(path, "training.1600000.processed.noemoticon.csv")

cols = ["target", "id", "date", "flag", "user", "text"]
df = pd.read_csv(
    csv_path,
    encoding="latin-1",
    names=cols
)

# Labels are 0 (negative) and 4 (positive). Remap to 0/1.
df["target"] = df["target"].map({0: 0, 4:1})

df = df[["target", "text"]]
print(df.shape)
print(df["target"].value_counts())
df.head()

(1600000, 2)
target
0    800000
1    800000
Name: count, dtype: int64


,target,text
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,is upset that he can't update his Facebook by ...
2,0,@Kenichan I dived many times for the ball. Man...
3,0,my whole body feels itchy and like its on fire
4,0,"@nationwideclass no, it's not behaving at all...."


# PREPROCESSING

In [ ]:
import re
import html

def clean_tweet(text):
    text = html.unescape(text)                     # &amp; -> &, &quot; -> "
    text = re.sub(r"https?://\S+|www\.\S+", "", text)  # URLs
    text = re.sub(r"@\w+", "", text)               # mentions
    text = re.sub(r"\s+", " ", text).strip()       # extra whitespace
    return text

df["clean_text"] = df["text"].apply(clean_tweet)

# Check results
print(df[["text", "clean_text"]].head())

# Some tweets may become empty after cleaning, so drop them
print((df["clean_text"] == "").sum(), "empty tweets")
df = df[df["clean_text"] != ""].reset_index(drop=True)

                                                text  \
0  @switchfoot http://twitpic.com/2y1zl - Awww, t...   
1  is upset that he can't update his Facebook by ...   
2  @Kenichan I dived many times for the ball. Man...   
3    my whole body feels itchy and like its on fire    
4  @nationwideclass no, it's not behaving at all....   

                                          clean_text  
0  - Awww, that's a bummer. You shoulda got David...  
1  is upset that he can't update his Facebook by ...  
2  I dived many times for the ball. Managed to sa...  
3     my whole body feels itchy and like its on fire  
4  no, it's not behaving at all. i'm mad. why am ...  
2815 empty tweets


### Tokenization and Padding


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_LEN = 28
VOCAB_SIZE = 50000   # top words kept, rest become <OOV>
# 1. Split first: 80% train, 10% val, 10% test (stratified)
X_train, X_temp, y_train, y_temp = train_test_split(
    df["clean_text"], df["target"], test_size=0.2,
    stratify=df["target"], random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

# 2. Fit tokenizer on TRAIN only
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>", filters="")
tokenizer.fit_on_texts(X_train)   # lowercases by default

# 3. Convert to sequences and pad/truncate to 28
def encode(texts):
    seqs = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=MAX_LEN, padding="post", truncating="post")

X_train_pad = encode(X_train)
X_val_pad   = encode(X_val)
X_test_pad  = encode(X_test)

y_train, y_val, y_test = y_train.values, y_val.values, y_test.values

print(X_train_pad.shape, X_val_pad.shape, X_test_pad.shape)
print("Vocab found in train:", len(tokenizer.word_index))
print(X_train_pad[0])

(1277748, 28) (159718, 28) (159719, 28)
Vocab found in train: 660466
[   99   511    86   117    82    83   395     1     5     1    82    45
    12   154   598    82     1   805 19962     0     0     0     0     0
     0     0     0     0]


# THE TWO MODELS

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout,GRU,SpatialDropout1D
import numpy as np


In [ ]:
!pip install -q keras-tuner

In [ ]:

import keras_tuner as kt
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

def make_builder(rnn_type):
    RNN = layers.LSTM if rnn_type == "lstm" else layers.GRU

    def build_model(hp):
        emb_dim   = hp.Choice("emb_dim", [50, 100, 200])
        units     = hp.Choice("units", [32, 64, 128])
        n_layers  = hp.Int("n_layers", 1, 3)
        dropout   = hp.Float("dropout", 0.2, 0.5, step=0.1)
        dense_u   = hp.Choice("dense_units", [32, 64])
        lr        = hp.Choice("lr", [1e-2, 3e-3, 1e-3, 3e-4])

        model = models.Sequential()
        model.add(layers.Embedding(VOCAB_SIZE, emb_dim, mask_zero=True))

        for i in range(n_layers):
            last = (i == n_layers - 1)
            model.add(layers.Bidirectional(RNN(units, return_sequences=not last)))
            model.add(layers.Dropout(dropout))

        model.add(layers.Dense(dense_u, activation="relu"))
        model.add(layers.Dropout(dropout))
        model.add(layers.Dense(1, activation="sigmoid"))

        model.compile(
            optimizer=tf.keras.optimizers.Adam(lr),
            loss="binary_crossentropy",
            metrics=["accuracy"],
        )
        return model

    return build_model


def run_tuner(rnn_type):
    tuner = kt.Hyperband(
        make_builder(rnn_type),
        objective="val_accuracy",
        max_epochs=8,
        factor=3,
        directory="tuner_results",
        project_name=f"bi{rnn_type}",
        seed=42,
        overwrite=True,
    )
    es = callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)
    tuner.search(
        X_train_pad, y_train,
        validation_data=(X_val_pad, y_val),
        batch_size=128,
        callbacks=[es],
    )
    return tuner

tuner_lstm = run_tuner("lstm")
tuner_gru  = run_tuner("gru")

for name, t in [("LSTM", tuner_lstm), ("GRU", tuner_gru)]:
    best = t.get_best_hyperparameters(1)[0]
    print(name, best.values)
    print(name, "best val_acc:", t.oracle.get_best_trials(1)[0].score)

In [ ]:
# lstm with best hyperparams and EarlyStopping
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping

model_lstm = Sequential([
    Embedding(VOCAB_SIZE, 200),
    SpatialDropout1D(0.3),
    Bidirectional(LSTM(32, return_sequences=False)),
    Dropout(0.3),
    Dense(32, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

model_lstm.compile(
    optimizer=tf.keras.optimizers.Adam(0.003),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

# Define early stopping to monitor validation loss
early_stop = EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)

history_lstm = model_lstm.fit(
    X_train_pad,
    y_train,
    epochs=8,
    batch_size=128,
    validation_data=(X_val_pad, y_val),
    callbacks=[early_stop]
)

Epoch 1/8
9983/9983 ━━━━━━━━━━━━━━━━━━━━ 2124s 212ms/step - accuracy: 0.8061 - loss: 0.4262 - val_accuracy: 0.8206 - val_loss: 0.3933
Epoch 2/8
7070/9983 ━━━━━━━━━━━━━━━━━━━━ 10:08 209ms/step - accuracy: 0.8369 - loss: 0.3721

In [ ]:
# GRU with best params and EarlyStopping
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping

model_gru = Sequential([
    Embedding(VOCAB_SIZE, 100),
    SpatialDropout1D(0.3),
    Bidirectional(GRU(32, return_sequences=True)),
    Dropout(0.2),
    Bidirectional(GRU(32, return_sequences=False)),
    Dropout(0.2),
    Dense(64, activation="relu"),
    Dropout(0.2),
    Dense(1, activation="sigmoid")
])

model_gru.compile(
    optimizer=tf.keras.optimizers.Adam(0.003),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

# Define early stopping to monitor validation loss
early_stop = EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)

history_gru = model_gru.fit(
    X_train_pad,
    y_train,
    epochs=5,
    batch_size=128,
    validation_data=(X_val_pad, y_val),
    callbacks=[early_stop]
)

In [ ]:
y_pred_gru = model_gru.predict(X_test_pad)
y_pred_gru = np.round(y_pred_gru).flatten().astype(int)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
classgru =  classification_report(y_test, y_pred_gru)
print(classgru)
accgru = accuracy_score(y_test, y_pred_gru)
print(accgru)

In [ ]:
y_pred_lstm = model_lstm.predict(X_test_pad)
y_pred_lstm = np.round(y_pred_lstm).flatten().astype(int)

In [ ]:


classlstm =  classification_report(y_test, y_pred_lstm)
print(classlstm)
acclstm = accuracy_score(y_test, y_pred_lstm)
print(acclstm)

In [ ]:
import matplotlib.pyplot as plt

# Set up the plotting grid
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Plot 1: Loss ---
if 'history_lstm' in globals() and history_lstm is not None:
    axes[0].plot(history_lstm.history['loss'], label='LSTM Train Loss', color='blue', linestyle='--')
    axes[0].plot(history_lstm.history['val_loss'], label='LSTM Val Loss', color='blue')

if 'history_gru' in globals() and history_gru is not None:
    axes[0].plot(history_gru.history['loss'], label='GRU Train Loss', color='orange', linestyle='--')
    axes[0].plot(history_gru.history['val_loss'], label='GRU Val Loss', color='orange')

axes[0].set_title('Training & Validation Loss')
axes[0].set_xlabel('Epochs')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True)

# --- Plot 2: Accuracy ---
if 'history_lstm' in globals() and history_lstm is not None:
    axes[1].plot(history_lstm.history['accuracy'], label='LSTM Train Acc', color='blue', linestyle='--')
    axes[1].plot(history_lstm.history['val_accuracy'], label='LSTM Val Acc', color='blue')

if 'history_gru' in globals() and history_gru is not None:
    axes[1].plot(history_gru.history['accuracy'], label='GRU Train Acc', color='orange', linestyle='--')
    axes[1].plot(history_gru.history['val_accuracy'], label='GRU Val Acc', color='orange')

axes[1].set_title('Training & Validation Accuracy')
axes[1].set_xlabel('Epochs')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
for name, m in [("LSTM", model_lstm), ("GRU", model_gru)]:
    t0 = time.time()
    m.predict(X_test_pad, batch_size=1024, verbose=0)
    print(f"{name}: {m.count_params():,} params, "
          f"inference {time.time() - t0:.1f}s on {len(X_test_pad):,} tweets")

In [ ]:

model_lstm.save("bilstm.keras")
model_gru.save("bigru.keras")

import pickle
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)